# NeuroBright: EEG Signal Recording and Visualization

This notebook guides you through:
1. Testing hardware connection
2. Recording baseline and all 3 brain states
3. Visualizing raw EEG signals
4. Analyzing frequency content and band powers
5. Preprocessing and creating the training dataset

## 1. Setup: Import Libraries and Load Configuration

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils.logger import get_logger, log_section_header
from src.utils.signal_utils import load_config, extract_band_powers
from src.data.collector import BrainDataCollector
from src.data.preprocessor import SignalPreprocessor

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Load configuration
config = load_config()
logger = get_logger(__name__)

print("Configuration loaded:")
print(f"  Channels: {config['hardware']['n_channels']}")
print(f"  Sample Rate: {config['hardware']['sample_rate']} Hz")
print(f"  Window Size: {config['windowing']['window_seconds']}s")
print(f"  Classes: {config['model']['class_names']}")

## 2. Test Hardware Connection

Verify that Arduino is connected and sending data correctly.

In [ ]:
# Initialize collector
collector = BrainDataCollector(config)

# Test connection
log_section_header("Hardware Connection Test")
if collector.test_connection():
    print("\n✓ Hardware connection successful!")
    print("✓ Ready to record data")
else:
    print("\n✗ Connection failed. Check COM port and Arduino.")

## 3. Record Baseline (30 seconds)

Record resting state with eyes open, sitting still.

In [ ]:
# Connect to Arduino
collector._connect()

# Record baseline
print("Prepare for baseline recording:")
print("  - Sit comfortably")
print("  - Eyes open")
print("  - Relax and stay still")
input("\nPress ENTER when ready...")

baseline = collector.record_baseline(duration=30)

print(f"\n✓ Baseline recorded: {baseline.shape}")

## 4. Record FOCUSED State (7 minutes)

Solve math problems continuously to maintain high engagement.

In [ ]:
# Get focused state configuration
focused_config = config['recording']['states']['focused']

print("FOCUSED STATE RECORDING")
print("="*60)
print(f"Duration: {focused_config['duration_seconds']}s ({focused_config['duration_seconds']//60} minutes)")
print(f"Task: {focused_config['instruction']}")
print(f"Examples: {', '.join(focused_config['task_examples'])}")
print("="*60)
input("\nPress ENTER to start recording...")

focused_signal, focused_label = collector.record_state(
    label=focused_config['label'],
    instruction=focused_config['instruction'],
    duration=focused_config['duration_seconds']
)

# Save to CSV
from datetime import datetime
session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_dir = Path(config['paths']['raw_data'])
raw_dir.mkdir(parents=True, exist_ok=True)

focused_file = raw_dir / f"neurobright_raw_focused_{session_id}.csv"
df_focused = pd.DataFrame(focused_signal, columns=config['hardware']['channel_names'])
df_focused['label'] = focused_label
df_focused.to_csv(focused_file, index=False)

print(f"\n✓ Saved to {focused_file}")

## 5. Record DROWSY State (7 minutes)

Sit still, breathe slowly, let eyes go heavy.

In [ ]:
# Get drowsy state configuration
drowsy_config = config['recording']['states']['drowsy']

print("\nTake a 3-minute break before next recording...")
import time
time.sleep(180)

print("\nDROWSY STATE RECORDING")
print("="*60)
print(f"Duration: {drowsy_config['duration_seconds']}s ({drowsy_config['duration_seconds']//60} minutes)")
print(f"Task: {drowsy_config['instruction']}")
print("="*60)
input("\nPress ENTER to start recording...")

drowsy_signal, drowsy_label = collector.record_state(
    label=drowsy_config['label'],
    instruction=drowsy_config['instruction'],
    duration=drowsy_config['duration_seconds']
)

# Save to CSV
drowsy_file = raw_dir / f"neurobright_raw_drowsy_{session_id}.csv"
df_drowsy = pd.DataFrame(drowsy_signal, columns=config['hardware']['channel_names'])
df_drowsy['label'] = drowsy_label
df_drowsy.to_csv(drowsy_file, index=False)

print(f"\n✓ Saved to {drowsy_file}")

## 6. Record STRESSED State (7 minutes)

Type as fast as possible with visible countdown timer.

In [ ]:
# Get stressed state configuration
stressed_config = config['recording']['states']['stressed']

print("\nTake a 3-minute break before next recording...")
time.sleep(180)

print("\nSTRESSED STATE RECORDING")
print("="*60)
print(f"Duration: {stressed_config['duration_seconds']}s ({stressed_config['duration_seconds']//60} minutes)")
print(f"Task: {stressed_config['instruction']}")
print("="*60)
input("\nPress ENTER to start recording...")

stressed_signal, stressed_label = collector.record_state(
    label=stressed_config['label'],
    instruction=stressed_config['instruction'],
    duration=stressed_config['duration_seconds']
)

# Save to CSV
stressed_file = raw_dir / f"neurobright_raw_stressed_{session_id}.csv"
df_stressed = pd.DataFrame(stressed_signal, columns=config['hardware']['channel_names'])
df_stressed['label'] = stressed_label
df_stressed.to_csv(stressed_file, index=False)

print(f"\n✓ Saved to {stressed_file}")

# Disconnect
collector._disconnect()
print("\n✓ All recordings complete!")

## 7. Load and Display Statistics

Load all saved recordings and show basic statistics.

In [ ]:
# Load all CSV files
csv_files = list(raw_dir.glob("*.csv"))
print(f"Found {len(csv_files)} recording files:\n")

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    label = df['label'].iloc[0]
    state_name = config['model']['class_names'][label]
    duration = len(df) / config['hardware']['sample_rate']
    
    print(f"  {csv_file.name}")
    print(f"    State: {state_name}")
    print(f"    Samples: {len(df)}")
    print(f"    Duration: {duration:.1f}s")
    print(f"    Mean amplitude: {df[config['hardware']['channel_names']].abs().mean().mean():.2f} µV")
    print()

## 8. Plot Raw EEG Signals (10 seconds each)

Visualize raw signals for each brain state.

In [ ]:
# Plot 10 seconds of each state
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
sample_rate = config['hardware']['sample_rate']
channels = config['hardware']['channel_names']
n_samples = sample_rate * 10  # 10 seconds

signals = [focused_signal, drowsy_signal, stressed_signal]
state_names = config['model']['class_names']

for idx, (signal, state_name, ax) in enumerate(zip(signals, state_names, axes)):
    # Take first 10 seconds
    segment = signal[:n_samples]
    time = np.arange(len(segment)) / sample_rate
    
    # Plot each channel
    for ch_idx, ch_name in enumerate(channels):
        ax.plot(time, segment[:, ch_idx], label=ch_name, alpha=0.7)
    
    ax.set_title(f"{state_name.upper()} State - Raw EEG (10s)", fontsize=14, fontweight='bold')
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude (µV)")
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Power Spectral Density Comparison

Compare frequency content across brain states using Welch's method.

In [ ]:
from scipy import signal as sp_signal

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ch_idx, ch_name in enumerate(channels):
    ax = axes[ch_idx]
    
    for sig, state_name in zip(signals, state_names):
        # Compute PSD
        freqs, psd = sp_signal.welch(
            sig[:, ch_idx],
            fs=sample_rate,
            nperseg=sample_rate * 2
        )
        
        # Plot only up to 50 Hz
        mask = freqs <= 50
        ax.semilogy(freqs[mask], psd[mask], label=state_name, alpha=0.7, linewidth=2)
    
    ax.set_title(f"Channel: {ch_name}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power Spectral Density")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Power Spectral Density Comparison", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Band Power Analysis

Extract and compare power in different frequency bands (Delta, Theta, Alpha, Beta, Gamma).

In [ ]:
# Extract band powers for each state
bands = config['bands']
band_names = list(bands.keys())

# Calculate average band powers
band_power_data = []

for sig, state_name in zip(signals, state_names):
    # Use middle 60 seconds
    start = len(sig) // 2 - sample_rate * 30
    end = start + sample_rate * 60
    segment = sig[start:end]
    
    powers = extract_band_powers(segment, sample_rate, bands)
    
    for band_name in band_names:
        avg_power = np.mean(powers[band_name])
        band_power_data.append({
            'State': state_name,
            'Band': band_name,
            'Power': avg_power
        })

df_bands = pd.DataFrame(band_power_data)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=df_bands, x='Band', y='Power', hue='State', ax=ax)
ax.set_title("Average Band Power by Brain State", fontsize=14, fontweight='bold')
ax.set_ylabel("Power (µV²)")
ax.set_xlabel("Frequency Band")
plt.xticks(rotation=0)
plt.legend(title='Brain State')
plt.tight_layout()
plt.show()

print("\nBand Power Summary:")
print(df_bands.pivot(index='Band', columns='State', values='Power'))

## 11. Spectrogram Visualization

Show time-frequency representation for each channel.

In [ ]:
# Plot spectrograms for focused state (first channel)
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (sig, state_name, ax) in enumerate(zip(signals, state_names, axes)):
    # Use first 60 seconds
    segment = sig[:sample_rate * 60, 0]  # First channel
    
    # Compute spectrogram
    f, t, Sxx = sp_signal.spectrogram(
        segment,
        fs=sample_rate,
        nperseg=sample_rate * 2,
        noverlap=sample_rate
    )
    
    # Plot only up to 50 Hz
    mask = f <= 50
    im = ax.pcolormesh(t, f[mask], 10 * np.log10(Sxx[mask]), shading='gouraud', cmap='viridis')
    ax.set_title(f"{state_name.upper()} State - Spectrogram (Channel {channels[0]})", 
                fontsize=12, fontweight='bold')
    ax.set_ylabel("Frequency (Hz)")
    ax.set_xlabel("Time (s)")
    plt.colorbar(im, ax=ax, label='Power (dB)')

plt.tight_layout()
plt.show()

## 12. Run Preprocessor

Apply filters, artifact rejection, and create windowed dataset.

In [ ]:
# Initialize preprocessor
preprocessor = SignalPreprocessor(config)

# Process full dataset
log_section_header("Signal Preprocessing")
X, y, statistics = preprocessor.process_full_dataset()

print("\n" + "="*60)
print("Preprocessing Complete!")
print("="*60)
print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Total windows: {statistics['total_windows']}")
print(f"Rejection rate: {statistics['rejection_rate']:.1f}%")
print(f"\nClass distribution:")
for class_idx, count in statistics['class_distribution'].items():
    state_name = config['model']['class_names'][class_idx]
    percentage = (count / statistics['total_windows']) * 100
    print(f"  {state_name}: {count} ({percentage:.1f}%)")

## 13. Visualize Processed Windows

Show examples of preprocessed windows for each class.

In [ ]:
# Plot example windows from each class
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for class_idx in range(3):
    # Find windows for this class
    class_mask = y == class_idx
    class_windows = X[class_mask]
    
    # Plot first window for each channel
    window = class_windows[0]  # (n_channels, window_samples)
    time = np.arange(window.shape[1]) / sample_rate
    
    for ch_idx in range(3):
        ax = axes[class_idx, ch_idx]
        ax.plot(time, window[ch_idx], linewidth=1)
        
        if ch_idx == 0:
            ax.set_ylabel(f"{state_names[class_idx].upper()}\nAmplitude", fontweight='bold')
        if class_idx == 0:
            ax.set_title(f"Channel {channels[ch_idx]}", fontweight='bold')
        if class_idx == 2:
            ax.set_xlabel("Time (s)")
        
        ax.grid(True, alpha=0.3)

plt.suptitle("Preprocessed Window Examples", fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 14. Dataset Summary

Final confirmation that processed data is ready for training.

In [ ]:
print("\n" + "="*60)
print("DATASET READY FOR TRAINING")
print("="*60)
print(f"\nProcessed data saved to: {config['paths']['processed_data']}")
print(f"  - X_windows.npy: {X.shape}")
print(f"  - y_labels.npy: {y.shape}")
print(f"\nNext steps:")
print("  1. Train model: python app.py --mode train")
print("  2. Or use notebook: 03_model_training.ipynb")
print("="*60)